# Synergy prediction using bulk RNA-seq data

## Data loading

In [1]:
import pandas as pd
import numpy as np
import os
import functools
from functools import reduce

Load the l2fc (compared to time zero) data and the CFU data for each condition.

In [2]:
import pandas as pd
import numpy as np
import os
import functools
from functools import reduce

def clean_csv_name(name):
    """
    Function to clean a csv name into a readable sample ID

    Args:
        name : ex: "T4-wt1AMX1hr-T4-wtNDC0hr-DGE-results"
    
    Returns:
        cleaned : ex: "1AMX1hr"
    """
    suffixes = [
        ".csv",
        "-T4-wtNDC0hr-DGE-results",
        "-T4-wtNDC1hr-DGE-results",
        "-T4-wtNDC2hr-DGE-results",
        "-T4-wtNDC4hr-DGE-results",
        "T4-wt"
    ]
    
    cleaned = name

    for suffix in suffixes:
        cleaned = cleaned.replace(suffix, "")

    return cleaned

def read_l2fc_as_df(data_dir, time_matched):
    """
    Function to extract all l2fc values from a directory of DGE results for various conditions.
    The output will be a list of dataframes, where each DGE result is stored as a dataframe with gene IDs on index, and 1 column of l2fc values

    Args:
        data_dir     : path to folder containing DGE results
        time_matched : boolean indicating whether DGE results should be from time zero or time-matched NDC

    Returns:
        l2fc_df_list : list of dataframes storing l2fc results
        names        : sample IDs corresponding to each dataframe
    """
    all_files = os.listdir(data_dir)
    files = []

    # Time-matched NDC comparisons
    if time_matched:
        for f in all_files:
            if f.endswith(".csv") and "NDC0hr" in f:
                files.append(f)\
    
    # Time zero comparisons
    else:
        for f in all_files:
            if f.endswith(".csv") and "NDC0hr" not in f:
                files.append(f)

    if len(files) == 0: 
        raise FileNotFoundError(f"No matching DGE files found in {data_dir}")

    # Sort and specify path to data files
    files = sorted(files)

    # Get sample IDs
    ids = [clean_csv_name(f) for f in files]

    # Convert csvs to dataframes
    filenames = ["".join([data_dir, "/" , f]) for f in files]
    l2fc_df_list = [pd.read_table(filenames[i], 
                                  sep = ",", 
                                  header = 0, 
                                  index_col = 1)["log2FoldChange"].rename(ids[i]) # Genes
                                  for i in range(len(filenames))]

    return l2fc_df_list, ids

def bind_l2fc_data(l2fc_df_list, ids):
    """
    Function to take a list of l2fc DEG dataframes, then bind all into 1 dataframe
    Args: 
        l2fc_df_list : list of l2fc dataframes
        ids          : corresponding sample IDs

    Returns:
        all_l2fc [N,G] : dataframe with all feature values (N samples on row, G genes on column)
    """
    all_l2fc = reduce(lambda df1, df2 :
                      pd.merge(df1, df2, 
                               left_index = True, 
                               right_index = True, 
                               how = "outer"),
                        l2fc_df_list)

    # Tranpose to get genes on columns
    all_l2fc = all_l2fc.T

    return all_l2fc


def read_avg_cfus(folder_path):
    """
    Function to extract CFUs into a dataframe with conditions on rows, 1 CFU column
    Args:
        folder_path : path to folder containing CFUs (same format as /all_cfus)

    Returns:
        all_avg_cfus [N,1] : df with condition names as index, 1 column of avg CFUs across triplicates (N = # samples)
    """

    # Get files
    files = os.listdir(folder_path)
    
    # Select CSV files
    cfu_files = [csv for csv in files if ".csv" in csv]
    cfu_files = sorted(cfu_files)
    cfu_files = ["".join([folder_path, "/", csv]) for csv in files]
    
    # Load each file as a dataframe
    cfu_dfs = [pd.read_table(csv, sep = ",", header = 0) for csv in cfu_files] 

    # change all of this to calculate average CFU for each condition
    for i, df in enumerate(cfu_dfs):

        # Remove the triplicate column
        df = df.drop(columns = "Triplicates")

        # Calculate mean of three triplicates and store
        means = df.mean()
        cfu_dfs[i] = means
    
    # Concat
    all_avg_cfus = pd.concat(cfu_dfs, axis = 0) # series
    all_avg_cfus = all_avg_cfus.rename("CFU").to_frame()

    # Remove space breaker characters
    all_avg_cfus.index = [id.replace("\xa0", "") for id in all_avg_cfus.index]
    
    return all_avg_cfus


def bind_all_data(feature_df, cfu_df):
    """
    Function bind TPM and cfu dfs
    Args:
        featurem_df [N,G] : Dataframe of TPMs, N = # samples, G = # genes, labels on index
        cfu_df [N,1]      : Dataframe of CFUs, labels on index

    Returns:
        data_df : [N, G+1] : Dataframe of all TPMs and CFUs as last column
    """
    # Right join so that CFUs exist
    data_df = pd.merge(feature_df, cfu_df, left_index = True, right_index = True, how = "right")
    data_df = data_df.iloc[data_df.index.argsort()]

    return data_df


def get_l2fc_and_cfu_data(l2fc_dir, cfu_dir, time_matched):
    
    l2fc_df_list, ids = read_l2fc_as_df(l2fc_dir, time_matched)
    all_l2fc = bind_l2fc_data(l2fc_df_list, ids)
    all_avg_cfus = read_avg_cfus(cfu_dir)
    data_df = bind_all_data(all_l2fc, all_avg_cfus)
    
    return data_df

Run the data extraction pipeline.

In [3]:
cfu_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/all_cfus"
l2fc_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/dge_timezero"

# Data pipeline
data_df = get_l2fc_and_cfu_data(l2fc_dir, cfu_dir, time_matched = True)

# Separate out the NDC CFU data and remove from the dataframe
cfus = data_df["CFU"]
ndc_ids = ["NDC0hr", "NDC1hr", "NDC2hr", "NDC4hr"]
ndc_cfus = [cfus.iloc[cfus.index == id].values[0] for id in ndc_ids]
ndc_df = pd.DataFrame(
    {"CFU": ndc_cfus},
    index = ndc_ids
)

# Remove the NDC columns from data df
data_df = data_df.iloc[[i for i in range(data_df.shape[0]) if data_df.index[i] not in ndc_ids]]

In [18]:
ndc_df

,CFU
NDC0hr,7.266667e+07
NDC1hr,7.933333e+08
NDC2hr,2.246667e+09
NDC4hr,2.400000e+10


Now, we can construct a metadata matrix that we will bind to the data. We will eventually have to calculate synergy scores and transcriptional interaction scores for all genes, so the annotated metatdata will help us match the approriate columns to calculate these. We will implement a few functions to help us do this.

In [ ]:
def condition_to_replicate_idx(condition_list):
    """
    Function to convert a list of conditions into replicate labels

    Args:
        condition_list : List of condition labels, ex: ["12CIP1hr-a", "12CIP1hr-b",...]

    Returns:
        idx_list : List of condition labels converted into replicate indices, ex: [0, 0, 1,...]
    """
    idx_list = [0]

    for i in range(1, len(condition_list)):
        prev_condition = condition_list[i-1]   
        curr_condition = condition_list[i]

        # Exclude last character since "-a" "-b"
        if prev_condition[:-1] == curr_condition[:-1]:
            idx_list.append(idx_list[i-1])
        
        else:
            idx_list.append(idx_list[i-1] + 1)
    
    return idx_list
    

def find_first_alpha(str):
    """
    Function to find the index of the the first letter in a string
    (will be used for drug name parser function)

    Args:
        str : string

    Returns:
        first_alpha : idx of first letter
    """
    for i in range(len(str)):
        if str[i].isalpha():
            return i
            break 


def condition_to_drug_id(condition_label):
    """
    Function to convert a condition label to a drug ID

    Args:
        condition_label : Condition label, ex: "12CEF1hr", "12CIP13VNC2hr"

    Returns:
        drug_id : Drug Id as a single string, ex: "CEF", "CIP+VNC"
    """
    # Count # of alpha characters
    num_upper = np.sum([char.isupper() for char in condition_label])

    # Single drug case
    if num_upper == 3:

        # Find drug name using letter search
        first_alpha_idx = find_first_alpha(condition_label)
        drug_id = condition_label[first_alpha_idx:first_alpha_idx + 3]
        return drug_id
    
    # Multiple drug case
    elif num_upper == 6:

        # Find first letter and extract drug name
        first_idx = find_first_alpha(condition_label)
        drug1 = condition_label[first_idx:first_idx + 3]

        # Find first letter in remaining string and extract drug name
        substr     = condition_label[first_idx + 3:]
        second_idx = find_first_alpha(substr)
        drug2      = substr[second_idx:second_idx + 3]

        # Merge drug names
        drug_id = drug1 + "+" + drug2

        return drug_id

    else:
        raise KeyError("Condition label does not fit drug ID criteria")


def condition_to_drugs(condition_label):
    """
    Function to convert a condition label to a list of the two drugs represented

    Args:
        condition_label : Condition label, ex: "12CEF1hr", "12CIP13VNC2hr"

    Returns:
        drugs : List of drugs (length 2, "" if 2nd drug doesn't exist), ex: ["CEF", NA], ["CIP", NA]
    """
    # Count # of alpha characters
    num_upper = np.sum([char.isupper() for char in condition_label])

    # Single drug case
    if num_upper == 3:

        # Find drug name using letter search
        first_alpha_idx = find_first_alpha(condition_label)
        drug = condition_label[first_alpha_idx:first_alpha_idx + 3]
        drugs = [drug, np.nan]
        return drugs
    
    # Multiple drug case
    elif num_upper == 6:

        # Find first letter and extract drug name
        first_idx = find_first_alpha(condition_label)
        drug1 = condition_label[first_idx:first_idx + 3]

        # Find first letter in remaining string and extract drug name
        substr = condition_label[first_idx + 3:]
        second_idx = find_first_alpha(substr)
        drug2 = substr[second_idx:second_idx + 3]
        drugs = [drug1, drug2]

        return drugs

    else:
        raise KeyError("Condition label does not fit drug ID criteria")
    

def condition_to_dose(condition_label):
    """
    Function to convert a condition label to a list of the two drugs represented

    Args:
        condition_label : Condition label, ex: "12CEF1hr", "12CIP13VNC2hr"

    Returns:
        doses : List of doses (length 2, "" if 2nd drug doesn't exist), ex: [0.50, NA], [0.50, 0.33]
    """
    # Define dose mapping
    dose_map = {
        "14": 0.25,
        "13": 0.33,
        "12": 0.50,
        "34": 0.75,
        "1": 1
    }

    # Count # of alpha characters
    num_upper = np.sum([char.isupper() for char in condition_label])

    # Single drug case
    if num_upper == 3:

        # Find drug name using letter search
        first_alpha_idx = find_first_alpha(condition_label)
        dose = condition_label[:first_alpha_idx]
        dose = dose_map[dose]
        doses = [dose, np.nan]
        return doses
    
    # Multiple drug case
    elif num_upper == 6:

        # Find first letter and extract drug name
        first_idx = find_first_alpha(condition_label)
        dose1 = condition_label[:first_idx]
        dose1 = dose_map[dose1]

        # Find first letter in remaining string and extract drug name
        substr = condition_label[first_idx + 3:]
        second_idx = find_first_alpha(substr)
        dose2 = substr[:second_idx]
        dose2 = dose_map[dose2]
        doses = [dose1, dose2]

        return doses

    else:
        raise KeyError("Condition label does not fit drug ID criteria")


def condition_to_timepoint(condition_label):
    """
    Function to convert a condition label to a timepoint

    Args:
        condition_label : Condition label, ex: "12CEF1hr"

    Returns:
        timepoint : Time, ex: 1
    """
    time_map = {
        "1hr": 1,
        "2hr": 2,
        "4hr": 4
    }

    # Find the timepoint label
    time_idx = condition_label.find("hr")
    timepoint = condition_label[time_idx - 1:]

    # Convert to int
    timepoint = time_map[timepoint]

    return timepoint

def condition_to_ndc_cfu(condition_label):

    

Now, we can apply these functions to construct a metadata matrix.

In [17]:
# Get condition ID names
labels = data_df.index

# Extract metadata from condition names
drug_id = [condition_to_drug_id(label) for label in labels]
drug = [condition_to_drugs(label) for label in labels]
drug1 = [x[0] for x in drug]
drug2 = [x[1] for x in drug]

dose = [condition_to_dose(label) for label in labels]
dose1 = [x[0] for x in dose]
dose2 = [x[1] for x in dose]

timepoint = [condition_to_timepoint(label) for label in labels]

# Add metadata columns to data df
data_df["drug_id"] = drug_id
data_df["drug1"] = drug1
data_df["drug2"] = drug2
data_df["drug1_dose"] = dose1
data_df["drug2_dose"] = dose2
data_df["timepoint"] = timepoint

# Add the baseline NDC CFUs for each timepoint


data_df.head(n = 5)

,F01-SP0085-SP0086,F02-SP0103-SP0104,F03-SP0115-SP0116,F04-SP0116-SP0117,F05-SP0117-SP0118,F06-SP0129-SP0130,F07-SP0239-SP0240,F08-SP0256-SP0257,F09-SP0257-SP0258,F10-SP0311-SP0312,...,SP_2238,SP_2239,SP_2240,CFU,drug_id,drug1,drug2,drug1_dose,drug2_dose,timepoint
12AMX1hr,0.203176,0.425140,0.421574,0.957388,0.318211,-1.596493,-0.889263,-0.828391,-0.472666,0.217035,...,-0.752067,-0.297218,0.162779,1.440000e+09,AMX,AMX,NaN,0.5,NaN,1
12AMX2hr,0.965367,0.148895,0.385028,0.636486,0.257198,-1.249236,-1.078364,-1.086718,-1.158409,-0.353335,...,-0.422076,-1.057505,-1.049118,3.493333e+09,AMX,AMX,NaN,0.5,NaN,2
12AMX4hr,-4.889168,0.411616,1.454771,-1.652460,1.519280,-2.430634,-0.610082,-0.623271,-2.740885,-0.228778,...,-0.519932,-1.004689,-0.692468,1.613333e+10,AMX,AMX,NaN,0.5,NaN,4
12CEF12CIP1hr,-0.160730,1.557105,-0.188761,0.115748,-1.231779,-0.077362,1.615068,1.723700,-1.129801,0.761837,...,0.622288,1.091430,0.537305,8.600000e+07,CEF+CIP,CEF,CIP,0.5,0.5,1
12CEF12CIP2hr,1.485571,1.617386,0.256227,-0.743481,-0.496051,1.671432,0.851560,0.584393,-0.251858,1.756313,...,0.839384,1.048523,1.009008,8.266667e+07,CEF+CIP,CEF,CIP,0.5,0.5,2
